# Updated Code

In [1]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 3.2.1
    Uninstalling sentence-transformers-3.2.1:
      Successfully uninstalled sentence-transformers-3.2.1


In [2]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from pathlib import Path
import pandas as pd
import re
import torch


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
data_path = Path("/content/drive/MyDrive/llama_3B_LR2e5_DR0_preds_final.xlsx")
df = pd.read_excel(data_path.as_posix())
print(df.shape)
df.head()

(750, 7)


,id,description,label,unstructured_preds,formatted_preds,json_preds,pydantic_preds
0,9dd929cc-072a-460d-8ad0-49ff9a356dca,"Position: SQL Server DBALocation: Plano, TX (D...","{""Required"":{""Education"":[],""Experience"":[{""in...","{""Required"":{""Education"":[],""Experience"":[{""in...","{\n ""Required"": {\n ""Education"": [],\n ...",NaN,"Required=Requirements(Education=[], Experience..."
1,bb0084db-a27a-493e-8341-dc09348dfbaa,"Title - OIC DeveloperLocation - Watsonville, C...","{""Required"":{""Education"":[{""field_of_study"":[""...","{""Required"":{""Education"":[{""field_of_study"":[""...","{\n ""Required"": {\n ""Education"": [\n ...",NaN,Required=Requirements(Education=[Education(edu...
2,783cf7ce-6ab4-438b-8db5-6a5e97b542f8,The Systems Engineer will support the Windows ...,"{""Required"":{""Education"":[{""field_of_study"":[""...","{""Required"":{""Education"":[{""field_of_study"":[""...","{\n ""Required"": {\n ""Education"": [\n ...",NaN,Required=Requirements(Education=[Education(edu...
3,41628488-45ac-401b-926c-020ca623658d,"Hi, I hope you're doing well !! Momento USA...","{""Required"":{""Education"":[],""Experience"":[],""C...","{""Required"":{""Education"":[],""Experience"":[{""in...","{\n ""Required"": {\n ""Education"": [],\n ...",NaN,"Required=Requirements(Education=[], Experience..."
4,ba33ae6b-7898-4c10-89f1-d418419f981b,Are you looking to elevate your cyber career? ...,"{""Required"":{""Education"":[{""field_of_study"":[]...","{""Required"":{""Education"":[{""field_of_study"":[]...","{\n ""Required"": {\n ""Education"": [\n ...",NaN,Required=Requirements(Education=[Education(edu...


In [6]:
df_test = df.copy()

df_test.isna().sum()

# Locate rows where 'unstructured_preds' is null
null_rows = df_test[df_test['json_preds'].isna()]

# Display the rows
print(null_rows)

                                       id  \
0    9dd929cc-072a-460d-8ad0-49ff9a356dca   
1    bb0084db-a27a-493e-8341-dc09348dfbaa   
2    783cf7ce-6ab4-438b-8db5-6a5e97b542f8   
3    41628488-45ac-401b-926c-020ca623658d   
4    ba33ae6b-7898-4c10-89f1-d418419f981b   
..                                    ...   
745  c791e041-2fe2-406e-970e-049ec27ddaa5   
746  d9fd4fa3-b508-4710-a584-a459ed632576   
747  8de99f45-76da-4517-9ef4-dad9f1a6d8a8   
748  6f240672-084c-484b-8095-c1c3be4c2644   
749  4f2663e3-b103-4336-a7f5-a471a2920c34   

                                           description  \
0    Position: SQL Server DBALocation: Plano, TX (D...   
1    Title - OIC DeveloperLocation - Watsonville, C...   
2    The Systems Engineer will support the Windows ...   
3    Hi,  I hope you're doing well   !! Momento USA...   
4    Are you looking to elevate your cyber career? ...   
..                                                 ...   
745  Are you interested in working in a dynamic env..

# Match Score with Precompute embeddings

In [7]:
import pandas as pd
import numpy as np
import json
import torch
from sentence_transformers import SentenceTransformer

# Load model on GPU if available or CPU otherwise
model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True, device='cuda' if torch.cuda.is_available() else 'cpu')


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/734k [00:00<?, ?B/s]

custom_st.py:   0%|          | 0.00/8.78k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/192 [00:00<?, ?B/s]

In [12]:
cat_cores = {
    'Technical_skills': 'skill_name',
    'Education': 'education_level',
    'Credentials': 'credential',
    'Experience': 'experience_desc'
}

def safe_get(d, keys):
    v = d
    for k in keys:
        v = v.get(k, [])
    return v

def compute_category_metrics(TP, FP, FN):
    if TP == 0 and FP == 0 and FN == 0:
        return 1.0, 1.0, 1.0
    precision = TP/(TP+FP) if (TP+FP)>0 else 0.0
    recall = TP/(TP+FN) if (TP+FN)>0 else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall)>0 else 0.0
    return precision, recall, f1

#############################
# Precompute Embeddings Logic
#############################

def extract_all_strings_from_json(j):
    # Recursively extract all strings from the JSON structure that are relevant for similarity checks
    # According to schema, main attributes are arrays of strings or integers.
    # We'll only embed strings.
    result = []
    if isinstance(j, dict):
        for v in j.values():
            result.extend(extract_all_strings_from_json(v))
    elif isinstance(j, list):
        for item in j:
            if isinstance(item, str):
                result.append(item)
            else:
                result.extend(extract_all_strings_from_json(item))
    return result

def precompute_embeddings_for_df(df, label_col='label_json', pred_col='prediction_json'):
    all_strings = set()
    # Extract from all rows
    for idx, row in df.iterrows():
        label_j = row[label_col]
        pred_j = row[pred_col]
        label_strings = extract_all_strings_from_json(label_j)
        pred_strings = extract_all_strings_from_json(pred_j)
        for s in label_strings:
            if s.strip():
                all_strings.add(s.strip())
        for s in pred_strings:
            if s.strip():
                all_strings.add(s.strip())
    all_strings = list(all_strings)

    # Compute embeddings in batch
    if all_strings:
        embeddings = model.encode(all_strings, batch_size=64)
    else:
        embeddings = np.zeros((0, 384)) # 384 is typical dimension for jina embeddings, may differ

    string_to_emb = {s: emb for s, emb in zip(all_strings, embeddings)}
    return string_to_emb

#############################
# Similarity Functions Using Precomputed Embeddings
#############################

def get_embedding(s, string_to_emb):
    # Return precomputed embedding or None if empty string
    s = s.strip()
    return string_to_emb.get(s, None)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b))

def text_similarity(text1, text2, string_to_emb, threshold=0.75):
    # If both empty
    if (not text1 or not text1.strip()) and (not text2 or not text2.strip()):
        return 1.0
    if (not text1 or not text1.strip()) or (not text2 or not text2.strip()):
        return 0.0
    emb1 = get_embedding(text1, string_to_emb)
    emb2 = get_embedding(text2, string_to_emb)
    if emb1 is None or emb2 is None:
        return 0.0
    sim = cosine_similarity(emb1, emb2)
    return sim if sim >= threshold else 0.0

def array_to_best_match_similarity(label_values, pred_values, string_to_emb, threshold=0.75):
    # label_values, pred_values are lists of strings
    # If both empty
    if (not label_values or all(not v.strip() for v in label_values)) and (not pred_values or all(not v.strip() for v in pred_values)):
        return 1.0
    if not label_values or all(not v.strip() for v in label_values):
        return 1.0
    if not pred_values or all(not v.strip() for v in pred_values):
        return 0.0

    # For each label_value, find best match in pred_values
    match_count = 0
    for lv in label_values:
        lv = lv.strip()
        if not lv:
            match_count += 1
            continue
        best_sim = 0.0
        for pv in pred_values:
            pv = pv.strip()
            if not pv:
                continue
            sim = text_similarity(lv, pv, string_to_emb, threshold=threshold)
            if sim > best_sim:
                best_sim = sim
        match_count += (best_sim > 0.0)
    return match_count / len([x for x in label_values if x.strip()])

def compute_requirement_match(label_req, pred_req, core_attr_name, string_to_emb, threshold=0.75):
    label_core = label_req.get(core_attr_name, [])
    pred_core = pred_req.get(core_attr_name, [])

    label_attrs = {k:v for k,v in label_req.items() if k != core_attr_name}
    pred_attrs = {k:v for k,v in pred_req.items() if k != core_attr_name}

    other_attr_keys = set(label_attrs.keys()) | set(pred_attrs.keys())
    num_other_attrs = len(other_attr_keys)
    other_attr_weight = 0.5 / num_other_attrs if num_other_attrs > 0 else 0.5

    # Core match
    core_match_fraction = array_to_best_match_similarity(label_core, pred_core, string_to_emb, threshold=threshold)
    core_score = core_match_fraction * 0.5

    # Other attributes match
    other_score = 0.0
    for attr in other_attr_keys:
        label_val = label_attrs.get(attr, [])
        pred_val = pred_attrs.get(attr, [])
        if not label_val and not pred_val:
            attr_match = 1.0
        else:
            # int arrays exact match, string arrays similarity
            if label_val and all(isinstance(x,int) for x in label_val):
                attr_match = 1.0 if set(label_val) == set(pred_val) else 0.0
            else:
                # treat as strings
                attr_match = array_to_best_match_similarity(label_val, pred_val, string_to_emb, threshold=threshold)
        other_score += attr_match * other_attr_weight

    total_score = core_score + other_score
    return total_score

#############################
# Matching Functions
#############################

def match_requirements_single_category(label_list, pred_list, core_attr_name, string_to_emb, threshold=0.75):
    if not label_list and not pred_list:
        return 0,0,0
    TP = 0.0
    matched_preds = set()
    for lreq in label_list:
        best_score = 0.0
        best_pred_idx = None
        for p_idx, preq in enumerate(pred_list):
            if p_idx in matched_preds:
                continue
            score = compute_requirement_match(lreq, preq, core_attr_name, string_to_emb, threshold=threshold)
            if score > best_score:
                best_score = score
                best_pred_idx = p_idx
        if best_pred_idx is not None and best_score > 0:
            TP += best_score
            matched_preds.add(best_pred_idx)

    FN = len(label_list) - len(matched_preds)
    FP = len(pred_list) - len(matched_preds)
    return TP, FP, FN

def match_requirements_with_cross(label_list, pred_list, core_attr,
                                  cross_pred_list, cross_core, string_to_emb, threshold=0.75):
    # Allows cross-category matches for Tech <-> Experience
    TP = 0.0
    matched_same_preds = set()
    matched_cross_preds = set()

    for lreq in label_list:
        best_same_score = 0.0
        best_same_idx = None
        for p_idx, preq in enumerate(pred_list):
            if p_idx in matched_same_preds:
                continue
            sc = compute_requirement_match(lreq, preq, core_attr, string_to_emb, threshold=threshold)
            if sc > best_same_score:
                best_same_score = sc
                best_same_idx = p_idx

        best_cross_score = 0.0
        best_cross_idx = None
        for p_idx, preq in enumerate(cross_pred_list):
            if p_idx in matched_cross_preds:
                continue
            sc = compute_requirement_match(lreq, preq, cross_core, string_to_emb, threshold=threshold)
            if sc > best_cross_score:
                best_cross_score = sc
                best_cross_idx = p_idx

        if best_cross_score > best_same_score:
            # use cross
            if best_cross_idx is not None:
                TP += best_cross_score
                matched_cross_preds.add(best_cross_idx)
        else:
            # use same
            if best_same_idx is not None and best_same_score > 0:
                TP += best_same_score
                matched_same_preds.add(best_same_idx)

    FN = len(label_list) - (len(matched_same_preds) + len(matched_cross_preds))
    if FN < 0: FN = 0
    FP = len(pred_list) - len(matched_same_preds)
    return TP, FP, FN, matched_same_preds, matched_cross_preds

def perform_tech_experience_global_matching(label_tech, pred_tech, label_exp, pred_exp, string_to_emb, threshold=0.75):
    # Match tech using experience as cross
    TTP, TFP, TFN, tech_matched_same, tech_matched_cross = match_requirements_with_cross(
        label_tech, pred_tech, cat_cores['Technical_skills'],
        pred_exp, cat_cores['Experience'], string_to_emb, threshold=threshold
    )

    # Remove used cross preds from experience before exp matching
    remaining_exp_preds = [p for i,p in enumerate(pred_exp) if i not in tech_matched_cross]

    # Match experience using tech as cross
    ETP, EFP, EFN, exp_matched_same, exp_matched_cross = match_requirements_with_cross(
        label_exp, remaining_exp_preds, cat_cores['Experience'],
        pred_tech, cat_cores['Technical_skills'], string_to_emb, threshold=threshold
    )

    # Remove used cross preds from tech
    remaining_tech_preds = [p for i,p in enumerate(pred_tech) if i not in exp_matched_cross]

    # Recalculate final metrics with cleaned preds
    final_TTP, final_TFP, final_TFN = match_requirements_single_category(label_tech, remaining_tech_preds, cat_cores['Technical_skills'], string_to_emb, threshold)
    final_ETP, final_EFP, final_EFN = match_requirements_single_category(label_exp, remaining_exp_preds, cat_cores['Experience'], string_to_emb, threshold)

    return (final_TTP, final_TFP, final_TFN, remaining_tech_preds,
            final_ETP, final_EFP, final_EFN, remaining_exp_preds)

def compute_all_metrics_for_row(label_json, pred_json, string_to_emb, threshold=0.75):
    # Extract combined sets
    def get_all(cat):
        return (safe_get(label_json, ['Required', cat]) + safe_get(label_json, ['Preferred', cat]),
                safe_get(pred_json, ['Required', cat]) + safe_get(pred_json, ['Preferred', cat]))

    (label_tech, pred_tech) = get_all('Technical_skills')
    (label_exp, pred_exp) = get_all('Experience')
    (label_edu, pred_edu) = get_all('Education')
    (label_cred, pred_cred) = get_all('Credentials')

    # Global match for tech & exp
    (final_TTP, final_TFP, final_TFN, remaining_tech_preds,
     final_ETP, final_EFP, final_EFN, remaining_exp_preds) = perform_tech_experience_global_matching(
        label_tech, pred_tech, label_exp, pred_exp, string_to_emb, threshold=threshold
    )

    # Education & Credentials direct match
    TP_edu, FP_edu, FN_edu = match_requirements_single_category(label_edu, pred_edu, cat_cores['Education'], string_to_emb, threshold)
    TP_cred, FP_cred, FN_cred = match_requirements_single_category(label_cred, pred_cred, cat_cores['Credentials'], string_to_emb, threshold)

    # Rebuild section-based sets:
    def get_section(cat, section):
        return safe_get(label_json, [section, cat]), safe_get(pred_json, [section, cat])

    # Filter back predictions for tech & exp:
    req_tech_pred = safe_get(pred_json, ['Required','Technical_skills'])
    pref_tech_pred = safe_get(pred_json, ['Preferred','Technical_skills'])

    def filter_remaining(original_list, remaining_list):
        rem_copy = remaining_list[:]
        filtered = []
        for o in original_list:
            found_idx = None
            for i,rc in enumerate(rem_copy):
                if rc == o:
                    found_idx = i
                    break
            if found_idx is not None:
                filtered.append(o)
                del rem_copy[found_idx]
        return filtered

    final_req_tech_pred = filter_remaining(req_tech_pred, remaining_tech_preds)
    final_pref_tech_pred = filter_remaining(pref_tech_pred, remaining_tech_preds)

    req_exp_pred = safe_get(pred_json, ['Required','Experience'])
    pref_exp_pred = safe_get(pred_json, ['Preferred','Experience'])
    final_req_exp_pred = filter_remaining(req_exp_pred, remaining_exp_preds)
    final_pref_exp_pred = filter_remaining(pref_exp_pred, remaining_exp_preds)

    req_tech_label, _ = get_section('Technical_skills','Required')
    pref_tech_label, _ = get_section('Technical_skills','Preferred')
    TP_req_tech, FP_req_tech, FN_req_tech = match_requirements_single_category(req_tech_label, final_req_tech_pred, cat_cores['Technical_skills'], string_to_emb, threshold)
    TP_pref_tech, FP_pref_tech, FN_pref_tech = match_requirements_single_category(pref_tech_label, final_pref_tech_pred, cat_cores['Technical_skills'], string_to_emb, threshold)

    req_exp_label, _ = get_section('Experience','Required')
    pref_exp_label, _ = get_section('Experience','Preferred')
    TP_req_exp, FP_req_exp, FN_req_exp = match_requirements_single_category(req_exp_label, final_req_exp_pred, cat_cores['Experience'], string_to_emb, threshold)
    TP_pref_exp, FP_pref_exp, FN_pref_exp = match_requirements_single_category(pref_exp_label, final_pref_exp_pred, cat_cores['Experience'], string_to_emb, threshold)

    req_edu_label, req_edu_pred = get_section('Education','Required')
    TP_req_edu, FP_req_edu, FN_req_edu = match_requirements_single_category(req_edu_label, req_edu_pred, cat_cores['Education'], string_to_emb, threshold)
    pref_edu_label, pref_edu_pred = get_section('Education','Preferred')
    TP_pref_edu, FP_pref_edu, FN_pref_edu = match_requirements_single_category(pref_edu_label, pref_edu_pred, cat_cores['Education'], string_to_emb, threshold)

    req_cred_label, req_cred_pred = get_section('Credentials','Required')
    TP_req_cred, FP_req_cred, FN_req_cred = match_requirements_single_category(req_cred_label, req_cred_pred, cat_cores['Credentials'], string_to_emb, threshold)
    pref_cred_label, pref_cred_pred = get_section('Credentials','Preferred')
    TP_pref_cred, FP_pref_cred, FN_pref_cred = match_requirements_single_category(pref_cred_label, pref_cred_pred, cat_cores['Credentials'], string_to_emb, threshold)

    # Aggregate
    TP_tech_total = TP_req_tech + TP_pref_tech
    FP_tech_total = FP_req_tech + FP_pref_tech
    FN_tech_total = FN_req_tech + FN_pref_tech

    TP_exp_total = TP_req_exp + TP_pref_exp
    FP_exp_total = FP_req_exp + FP_pref_exp
    FN_exp_total = FN_req_exp + FN_pref_exp

    TP_edu_total = TP_req_edu + TP_pref_edu
    FP_edu_total = FP_req_edu + FP_pref_edu
    FN_edu_total = FN_req_edu + FN_pref_edu

    TP_cred_total = TP_req_cred + TP_pref_cred
    FP_cred_total = FP_req_cred + FP_pref_cred
    FN_cred_total = FN_req_cred + FN_pref_cred

    TP_req_all = TP_req_tech + TP_req_edu + TP_req_cred + TP_req_exp
    FP_req_all = FP_req_tech + FP_req_edu + FP_req_cred + FP_req_exp
    FN_req_all = FN_req_tech + FN_req_edu + FN_req_cred + FN_req_exp

    TP_pref_all = TP_pref_tech + TP_pref_edu + TP_pref_cred + TP_pref_exp
    FP_pref_all = FP_pref_tech + FP_pref_edu + FP_pref_cred + FP_pref_exp
    FN_pref_all = FN_pref_tech + FN_pref_edu + FN_pref_cred + FN_pref_exp

    TP_total = TP_tech_total + TP_edu_total + TP_cred_total + TP_exp_total
    FP_total = FP_tech_total + FP_edu_total + FP_cred_total + FP_exp_total
    FN_total = FN_tech_total + FN_edu_total + FN_cred_total + FN_exp_total

    metrics = {}
    def add_metrics(prefix, TP, FP, FN):
        p,r,f = compute_category_metrics(TP,FP,FN)
        metrics[f"{prefix}_precision"] = p
        metrics[f"{prefix}_recall"] = r
        metrics[f"{prefix}_f1"] = f

    add_metrics("total", TP_total, FP_total, FN_total)
    add_metrics("required", TP_req_all, FP_req_all, FN_req_all)
    add_metrics("preferred", TP_pref_all, FP_pref_all, FN_pref_all)
    add_metrics("technical_skills", TP_tech_total, FP_tech_total, FN_tech_total)
    add_metrics("education", TP_edu_total, FP_edu_total, FN_edu_total)
    add_metrics("credentials", TP_cred_total, FP_cred_total, FN_cred_total)
    add_metrics("experience", TP_exp_total, FP_exp_total, FN_exp_total)
    add_metrics("required_technical_skills", TP_req_tech, FP_req_tech, FN_req_tech)
    add_metrics("required_education", TP_req_edu, FP_req_edu, FN_req_edu)
    add_metrics("required_credentials", TP_req_cred, FP_req_cred, FN_req_cred)
    add_metrics("required_experience", TP_req_exp, FP_req_exp, FN_req_exp)
    add_metrics("preferred_technical_skills", TP_pref_tech, FP_pref_tech, FN_pref_tech)
    add_metrics("preferred_education", TP_pref_edu, FP_pref_edu, FN_pref_edu)
    add_metrics("preferred_credentials", TP_pref_cred, FP_pref_cred, FN_pref_cred)
    add_metrics("preferred_experience", TP_pref_exp, FP_pref_exp, FN_pref_exp)

    return metrics

def compute_scores_for_df(df, label_col='label_json', pred_col='prediction_json', threshold=0.75):
    # Precompute embeddings for all strings in DF
    string_to_emb = precompute_embeddings_for_df(df, label_col=label_col, pred_col=pred_col)

    # List of all 15 category prefixes as defined in the original requirements
    categories = [
        "total",
        "required",
        "preferred",
        "technical_skills",
        "education",
        "credentials",
        "experience",
        "required_technical_skills",
        "required_education",
        "required_credentials",
        "required_experience",
        "preferred_technical_skills",
        "preferred_education",
        "preferred_credentials",
        "preferred_experience"
    ]

    # We need precision, recall, f1 for each category, so 45 keys
    metric_keys = []
    for cat in categories:
        metric_keys.append(f"{cat}_precision")
        metric_keys.append(f"{cat}_recall")
        metric_keys.append(f"{cat}_f1")

    # Function to return zero metrics if prediction_json is null
    def zero_metrics():
        return {k:0.0 for k in metric_keys}

    def process_row(row):
        if row[pred_col] is None or (isinstance(row[pred_col], float) and np.isnan(row[pred_col])):
            # prediction_json is null
            return zero_metrics()
        else:
            return compute_all_metrics_for_row(row[label_col], row[pred_col], string_to_emb, threshold=threshold)

    all_results = df.apply(process_row, axis=1)
    results_df = pd.DataFrame(all_results.tolist(), index=df.index)
    df = pd.concat([df, results_df], axis=1)
    return df

def aggregate_scores(df):
    metric_cols = [c for c in df.columns if c.endswith('_precision') or c.endswith('_recall') or c.endswith('_f1')]
    agg = df[metric_cols].mean()
    return agg.to_dict()

# Usage:
# df['label_json'] = df['label_json'].apply(json.loads)
# df['prediction_json'] = df['prediction_json'].apply(json.loads)
# df = compute_scores_for_df(df)
# final_scores = aggregate_scores(df)
# print(final_scores)


In [ ]:
# Example usage:
# Assuming your DataFrame is called `df` and has columns 'label_json' and 'prediction_json' with proper JSONs.
df_test['label_json'] = df_test['label'].apply(json.loads)
# Safely parse JSON with error handling
def safe_json_loads(val):
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):  # Catch JSON errors and type errors
        return None

df_test['prediction_json'] = df_test['unstructured_preds'].apply(safe_json_loads)
#df_test = df_test.dropna(subset=['label_json', 'prediction_json']).copy()
print(f"Number of Nulls resulting in 0 scores: {df_test['prediction_json'].isna().sum()}\n")
df_test = compute_scores_for_df(df_test)
final_scores = aggregate_scores(df_test)
print(final_scores)

Number of Nulls resulting in 0 scores: 3



In [29]:
boosting_df = {}
for index, row in df_test.iterrows():
    f1 = row['total_f1']
    if f1[0] < 0.5:
        boosting_df[index] = row
print(f"Number of rows with total_f1 > 0.5: {len(boosting_df)}")

Number of rows with total_f1 > 0.5: 293


<ipython-input-29-29513329a914>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if f1[0] < 0.5:


In [30]:
#convert boosting_df back to df, and save as excel
boosting_df = pd.DataFrame(boosting_df).T
boosting_df.to_excel('boosting_df.xlsx')